# 12 · Dative SMILES: writing a complex down, and reading it back

A complex has to survive being written to a string and read back, or nothing downstream can key on it.
Three verbs, all core (`numpy` and `rdkit`, no extra):

- `parse_smiles(text)` reads a SMILES, including a cxSMILES carrying an `atomProp` block
- `dative_smiles(mol)` writes the constitution, every M–donor bond normalised to the ionic dative form
- `canonical_smiles(source)` writes that plus the arrangement, as `<core> |atomProp:…|`

[`07_metal`](07_metal.ipynb) is the coordination sphere itself. This is the string it is written in, and the
two properties that make the string worth having: one species gives one string whichever Lewis form
described it, and the isomer the string names is the isomer the embed hands back.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Geometry import Point3D

import rxembed as rx

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

cisplatin = Chem.AddHs(Chem.MolFromSmiles("[NH3]->[Pt](<-[NH3])(Cl)Cl"))  # covalent Cl, as a perceiver draws it
print(rx.dative_smiles(cisplatin))

## Two layers, because SMILES can state only one of them

`dative_smiles` is the constitution: connectivity, charges and ligand stereocentres. cis- and transplatin
give one string here, the way `MolToSmiles(isomericSmiles=False)` gives one string for R and S. The
arrangement is left out rather than half-stated: RDKit would write it as an `@SP…`/`@OH…` tag, but that tag
is a permutation index over the neighbour *order*, so symmetry-equivalent ligands destabilise it (Fe(CO)₅
writes `@TB20`, `@TB14` or `@TB13` for one molecule), and OpenSMILES has a chirality class for 3 of the 12
rxembed polyhedra that carry geometric isomerism.

`canonical_smiles` states it for all of them a layer up, in a cxSMILES `atomProp` block: the 3-letter
geometry code and Λ/Δ on the metal, a canonical slot `s<n>` on each donor. Everything before the first `|`
is still the constitution, so `text.split('|', 1)[0]` is a species key any RDKit pipeline reads.

In [ ]:
isos = rx.enumerate_isomers(cisplatin, "square_planar")
for iso in isos:
    print(f"{iso.label:6s} |atomProp:{rx.canonical_smiles(iso).split('|atomProp:')[1]}")

cores = {rx.canonical_smiles(i).split(" |", 1)[0] for i in isos}
print(f"\n{len(isos)} arrangements, {len(cores)} constitution:", cores == {rx.dative_smiles(cisplatin)})

## Canonical over the species, not over the input

An M–L bond order is whatever the perceiver drew. The same perceiver hands back a terminal oxo SINGLE at
1.644 Å and an aqua DATIVE at 2.506 Å, and two different perceivers disagree by more than that, so a string
that inherited the bond order would not be canonical at all. `dative_smiles` decides from the DONOR's
valence instead, which is a fact about the element: a chloride has nothing but the metal to fill its shell,
so it takes the charge and donates, while a phosphine is already full and donates with no charge to move.

Each pair below is one species written two ways, with the total charge matched inside the pair: a covalent
`[Pd]Cl` is Pd(II)Cl₂ only against `[Cl-]->[Pd+2]`.

In [ ]:
one_species = {
    "chloride": ("[NH3]->[Pt](<-[NH3])(Cl)Cl", "[NH3]->[Pt+2](<-[NH3])(<-[Cl-])<-[Cl-]"),
    "phosphine + anion": ("CP(C)(C)->[Rh]Cl", "CP(C)(C)->[Rh+]<-[Cl-]"),
    "terminal oxo": ("O=[V](Cl)(Cl)Cl", "[O-2]->[V+5](<-[Cl-])(<-[Cl-])<-[Cl-]"),
}
for kind, pair in one_species.items():
    written = {rx.dative_smiles(Chem.AddHs(Chem.MolFromSmiles(s))) for s in pair}
    print(f"{kind:18s} {len(written)} string   {written.pop()}")

Where the rule stops, stated rather than papered over. An amide N and a methyl C are partly filled, so
both Lewis forms are legal SMILES for a neutral donor and neither is a repair the writer is entitled to
make. The charge that would merge them is exactly the Kekulé artefact this project has twice reverted from
the M–L distance model, where a formal charge split acac's two equivalent oxygens by 0.19 Å.

In [ ]:
partly_filled = {
    "amide": ("CN(C)[Pd](Cl)Cl", "C[N-](C)->[Pd+3](<-[Cl-])<-[Cl-]"),
    "alkyl": ("C[Pd](Cl)(Cl)C", "[CH3-]->[Pd+4](<-[Cl-])(<-[Cl-])<-[CH3-]"),
}
for kind, pair in partly_filled.items():
    written = {rx.dative_smiles(Chem.AddHs(Chem.MolFromSmiles(s))) for s in pair}
    print(f"{kind:6s} {len(written)} strings")
    for s in sorted(written):
        print("      ", s)

## Two doors, one constitution

A complex reaches the writer either as a `Mol` or as an `Isomer`, and an `Isomer` carries its metal as the
bond-less surrogate every embed stage needs, so writing one has to rebuild the sphere first. That rebuild is
where a Lewis form gets invented. Restoring every M–donor bond dative preserves the input's charges, which
is what a calculator wants and the opposite of what a canonical string wants: it is the third line of each
block below, a Pt(0) holding two neutral chlorides from the same molecule the direct door writes as `[Pt+2]`
with two `[Cl-]`. The sigma donors go back SINGLE instead and the valence rule re-derives the ionic form, as
it does for a `Mol`; before that, the two doors disagreed on 8 of the 45 corpus structures.

In [ ]:
from rxembed.metal_core import connect_metal  # the rebuild verb, here to show the form that was rejected

for smi in one_species["chloride"]:
    mol = Chem.AddHs(Chem.MolFromSmiles(smi))
    iso = rx.enumerate_isomers(mol)[0]
    print(f"in            {smi}")
    print(f"  Mol door    {rx.dative_smiles(mol)}")
    print(f"  Isomer door {rx.canonical_smiles(iso).split(' |', 1)[0]}")
    print(f"  all-dative  {Chem.MolToSmiles(connect_metal(iso.restore(Chem.Mol(iso.mol)), iso.donor_bonds))}")

## The round trip closes

The block indexes atoms by their POSITION in the string the writer just emitted, so the parse has to keep
the hydrogens the writer counted. `Chem.MolFromSmiles` strips explicit hydrogens and renumbers underneath
them, and every index in the block then addresses a different atom. `rx.parse_smiles` is the one SMILES door
for both tiers and keeps them when a block is present, which is why the round trip needs no second verb:
`enumerate_isomers` seats the arrangement the string states instead of enumerating.

In [ ]:
text = rx.canonical_smiles(isos[0])  # the cis isomer
back = rx.enumerate_isomers(rx.parse_smiles(text))
print(f"{len(back)} isomer read back: {back[0].label} | {back[0].summary()}")
print("fixed point:", rx.canonical_smiles(back[0]) == text)

naive = Chem.MolFromSmiles(text)  # the default parse: same string, six hydrogens short
print(f"\nparse_smiles {rx.parse_smiles(text).GetNumAtoms()} atoms | MolFromSmiles {naive.GetNumAtoms()} atoms")

plain = Chem.MolFromSmiles("[NH3]->[Pt](<-[NH3])(Cl)Cl")  # no block: nothing stated, so it enumerates
print("a plain SMILES still enumerates:", len(rx.enumerate_isomers(plain, "square_planar")), "isomers")

## The isomer the string names is the isomer you get

A stated arrangement the embed was free to mirror would be worth nothing, and mirroring is exactly what the
embed does when left alone: the coordination constraints are distances, angles, pulls, floors and a
coplanarity window symmetric about 180°, every one of them mirror-invariant, and the surrogate metal is
bond-less, so ETKDG has no stereocentre to enforce. The hand used to be whatever the seed fell on, with
41.6% of conformers coming back the input's enantiomer over `benchmark/corpus`. `minimize` reads the hand
off the relaxed sphere and reflects or re-seeds; the INFO line says how many of each, and `.wrong_hand`
lists the ones neither could fix.

Each conformer is checked below by writing its geometry back out. `canonical_smiles` on a `Mol` with a
conformer measures the arrangement off the coordinates rather than taking the `Isomer`'s word for it, so
`CXSMILES -> Isomer -> embed -> CXSMILES` is a fixed point rather than an approximation.

In [ ]:
en = Chem.AddHs(Chem.MolFromSmiles("Cl[Co]12(Cl)(<-NCC<-N1)<-NCC<-N2"))  # [Co(en)2Cl2]: a cis Δ/Λ pair
hands = [i for i in rx.enumerate_isomers(en, "octahedral") if i.chirality]

for iso in hands:
    stated = rx.canonical_smiles(iso)
    confs = rx.embed(iso, n=4, seed=1).minimize()
    measured = [rx.canonical_smiles(confs[k].mol) == stated for k in range(len(confs.ids))]
    print(
        f"{iso.chirality:7s} {sum(measured)}/{len(measured)} conformers measure back to their own string"
        f" | wrong_hand {confs.wrong_hand}"
    )

The block is what carries the hand, and a reflection moves the metal's note and nothing else: a slot is
a fact about the molecule modulo the template's *proper* rotations only, so the full point group is
proper × Z₂ and that Z₂ is the handedness. Negate one coordinate of the Λ geometry just embedded and the
word flips while every `s<n>` stays where it was.

In [ ]:
mirror = confs[0].mol  # one conformer: the Λ hand as it came out of the relax
conf = mirror.GetConformer()
pos = conf.GetPositions()
pos[:, 0] *= -1  # a reflection: one coordinate negated, nothing else touched
for a in range(mirror.GetNumAtoms()):
    conf.SetAtomPosition(a, Point3D(*pos[a]))

print("as embedded:", rx.canonical_smiles(confs[0].mol).split("|atomProp:")[1])
print("mirrored   :", rx.canonical_smiles(mirror).split("|atomProp:")[1])

## None of that needed an extra

Everything above is `numpy` and `rdkit`: a string in, an `Isomer`, an embed, a string out. Nothing has
reached `rxembed.pipeline`.

A `sys.modules` probe only shows that nothing *happened* to import it. That nothing *can* need it is checked
in `tests/test_metal_smiles.py`, which refuses `rxembed.pipeline`, `xyzgraph`, `scipy` and five other wheels
outright through `sys.meta_path`, runs this same round trip in a subprocess, and fails if the block turns
out not to be blocking.

In [ ]:
import sys

print("rxembed.pipeline imported:", "rxembed.pipeline" in sys.modules)

## Where the boundary is

Coordinates. Turning an `.xyz` into a graph is perception, which is `pipeline/perceive.py` and the
`perceive` extra; everything after it is core, including measuring the arrangement back off the geometry. So
a DFT or crystal structure becomes a species key in two steps and only the first needs the optional half.
`geom_check` is optional too, which is why the gate on the complexes embedded above lands here.

The Ru pincer below closes the same loop from the other end: `.xyz -> Mol -> CXSMILES -> Isomer -> CXSMILES`.
Two of its donors share slot `s1`, because its η² face is one vertex held by two atoms.

In [ ]:
from rxembed.pipeline import geom_check, read_xyz

rep = geom_check.check(confs.mol, conf_id=confs.ids[0], donors=list(hands[1].donors))
print("geometry.check:", "CLEAN" if rep.ok() else f"FAIL {rep.violations[:4]}")

ru = read_xyz("structures/ru-co.xyz")  # coordinates only: this is the step that needs the extra
ru_text = rx.canonical_smiles(ru)
core, block = ru_text.split(" |atomProp:")
seated = rx.enumerate_isomers(rx.parse_smiles(ru_text))[0]
print(f"\nru-co.xyz: {ru.GetNumAtoms()} atoms -> {len(core)} characters of constitution")
print(" ", block)
print(" ", seated.summary())
print("  fixed point:", rx.canonical_smiles(seated) == ru_text)